In [16]:
import error_sampling
from error_sampling import sample_errors_strict, sample_errors_cosine

In [17]:
import json

def load_run(run_id, run_dir="../prompt_runs"):
    with open(f"{run_dir}/{run_id}.json") as f:
        run_log = json.load(f)

    predictions_entities = {tuple(e) for e in run_log["extractions"]["entities"]}
    predictions_relationships = {tuple(r) for r in run_log["extractions"]["relations"]}

    return run_log, predictions_entities, predictions_relationships

In [ ]:
import pandas as pd

biored_train_gts = pd.read_csv("../data/filtered/biored_train_gts.csv")

ground_truth_entities = (
    set(zip(biored_train_gts["pmid"], biored_train_gts["entity_1"].str.strip().str.lower(),
        biored_train_gts["entity_1_type"]))
    | set(zip(biored_train_gts["pmid"], biored_train_gts["entity_2"].str.strip().str.lower(),
        biored_train_gts["entity_2_type"]))
)

ground_truth_relationships = set(zip(
    biored_train_gts["pmid"],
    biored_train_gts["entity_1"].str.strip().str.lower(),
    biored_train_gts["relation"],
    biored_train_gts["entity_2"].str.strip().str.lower()
))

In [20]:
from evaluation import build_embedding_lookup, relationship_match_cosine, entity_match_cosine

run_log, predictions_entities, predictions_relationships = load_run("run_006")

relationship_embeddings = build_embedding_lookup(predictions_relationships, ground_truth_relationships, text_indices=[1, 3])
entity_embeddings = build_embedding_lookup(predictions_entities, ground_truth_entities, text_indices=[1])

relation_fp_sample, relation_fn_sample = sample_errors_cosine(
    predictions_relationships, ground_truth_relationships, relationship_match_cosine, relationship_embeddings, n=20, label="Relationships"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Relationships: False Positives (predicted, no cosine match in ground truth) ---
Sampled 20 of 330 total FPs

  (17192049, 'ile462val', 'Association', 'prostate cancer')
  (20683499, 'stz', 'Cotreatment', 'crocin')
  (16120104, '111g allele', 'Positive_Correlation', 'extreme morning preference')
  (20510337, 'cisplatin', 'Negative_Correlation', 'superoxide dismutase')
  (25305591, 'foxp3', 'Association', 'tregs')
  (18808529, 'myocytolytic process', 'Negative_Correlation', 'beta-dystroglycan')
  (16574712, 'ecstasy', 'Positive_Correlation', 'memory updating')
  (15970799, 'slco1b1*15+c1007g', 'Negative_Correlation', 'cerivastatin')
  (15970799, 'slco1b1*15+c1007g', 'Negative_Correlation', 'estradiol-17beta-d-glucuronide')
  (17192049, 'cyp1a1', 'Association', 'environmental procarcinogens')
  (24477591, '-930a>g', 'Association', 'cyba')
  (20510337, 'cisplatin', 'Negative_Correlation', 'zinc ions')
  (20431083, 'cerebral microbleeds', 'Positive_Correlation', 'intracerebral hemorrhag